# Corpus Expansion v2 — Fixed Paths

English genres first: fiction, news, subtitles, TED.
Change PHASE for multilingual/FLORES.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, os, gc, random, time
from pathlib import Path
from scipy import stats
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/LRTIA')
RESULTS = DRIVE / 'Results/corpus_expansion/llama'
RESULTS.mkdir(parents=True, exist_ok=True)
TARGETS_PATH = DRIVE / 'Results/corpus_expansion/targets_llama.jsonl'
TOK_MANIFEST_PATH = DRIVE / 'Results/corpus_expansion/tokenized_manifest_llama.jsonl'
CORPUS_BASE = DRIVE / 'Data/corpus_expansion'

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'
C = 100
N_SHUFFLES = 1
SEED = 20260429

PHASE = 1
if PHASE == 1:
    RUN_CORPORA = ['gutenberg_fiction_en', 'news_en', 'subtitles_dialogue_en', 'ted_transcripts_en']
elif PHASE == 2:
    RUN_CORPORA = [
        'subtitles_dialogue_ar', 'subtitles_dialogue_de', 'subtitles_dialogue_es',
        'subtitles_dialogue_fr', 'subtitles_dialogue_ru', 'subtitles_dialogue_tr',
        'ted_transcripts_ar', 'ted_transcripts_de', 'ted_transcripts_es',
        'ted_transcripts_fr', 'ted_transcripts_ru', 'ted_transcripts_tr',
    ]
elif PHASE == 3:
    RUN_CORPORA = [f'flores_matched_{l}' for l in
        ['en','ar','de','es','fi','fr','he','hi','ja','ko','ru','sw','tr','vi','zh']]

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Phase {PHASE}: {RUN_CORPORA}')
print('Setup done')

In [ ]:
# === Load targets + tokenized manifest ===
all_targets = []
with open(TARGETS_PATH) as f:
    for line in f:
        all_targets.append(json.loads(line))
print(f'Targets: {len(all_targets)}')

targets_by_corpus = {}
for t in all_targets:
    targets_by_corpus.setdefault(t['corpus_id'], []).append(t)

tok_manifest = {}
with open(TOK_MANIFEST_PATH) as f:
    for line in f:
        d = json.loads(line)
        tok_manifest[d['document_id']] = d['file_path']
print(f'Tokenized manifest: {len(tok_manifest)} docs')

def to_drive_path(local_path):
    fname = '/'.join(str(local_path).replace('data/corpus_expansion/clean/', '').split('/'))
    return str(CORPUS_BASE / fname)

test_doc = list(tok_manifest.values())[0]
test_drive = to_drive_path(test_doc)
print(f'Path test: {test_drive}')
print(f'Exists: {Path(test_drive).exists()}')

for c in RUN_CORPORA:
    print(f'  {c}: {len(targets_by_corpus.get(c, []))} targets')

In [ ]:
# === Load model ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16),
    device_map='auto'
)
model.eval()
print('Llama loaded')

In [ ]:
# === PPL + corrected marginal functions ===

@torch.no_grad()
def ppl_nll(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2:
        return float('inf'), float('inf')
    full = list(ctx_toks) + list(tgt_toks)
    ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids)
    logits = out.logits[0]
    nll = 0.0
    cnt = 0
    for i in range(ts, len(full) - 1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll += -lp[full[i + 1]].item()
        cnt += 1
    del out, logits
    torch.cuda.empty_cache()
    if cnt == 0:
        return float('inf'), float('inf')
    mn = nll / cnt
    return math.exp(mn), mn


def compute_corrected_curves(ctx, tgt):
    mc = len(ctx)
    o_ppl = []
    o_nll = []
    s_ppl = []
    s_nll = []
    for c in range(mc + 1):
        pfx = ctx[-c:] if c > 0 else []
        p, n = ppl_nll(pfx, tgt)
        o_ppl.append(p)
        o_nll.append(n)
        if c == 0:
            s_ppl.append(p)
            s_nll.append(n)
        else:
            rng = random.Random(SEED + c)
            sp_l = []
            sn_l = []
            for _ in range(N_SHUFFLES):
                sh = list(pfx)
                rng.shuffle(sh)
                sp, sn = ppl_nll(sh, tgt)
                if not math.isinf(sp):
                    sp_l.append(sp)
                    sn_l.append(sn)
            s_ppl.append(np.mean(sp_l) if sp_l else p)
            s_nll.append(np.mean(sn_l) if sn_l else n)
    dists = list(range(1, mc + 1))
    mo = [o_ppl[d - 1] - o_ppl[d] for d in dists]
    ms = [s_ppl[d - 1] - s_ppl[d] for d in dists]
    delta = [a - b for a, b in zip(mo, ms)]
    mo_n = [o_nll[d - 1] - o_nll[d] for d in dists]
    ms_n = [s_nll[d - 1] - s_nll[d] for d in dists]
    delta_n = [a - b for a, b in zip(mo_n, ms_n)]
    return {
        'distances': dists,
        'ordered_ppl': o_ppl,
        'shuffled_ppl': s_ppl,
        'delta_ppl': delta,
        'ordered_nll': o_nll,
        'shuffled_nll': s_nll,
        'delta_nll': delta_n,
    }


BIN_EDGES = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]


def fit_pl(marg):
    bm = []
    bc = []
    for i in range(len(BIN_EDGES) - 1):
        lo = BIN_EDGES[i]
        hi = BIN_EDGES[i + 1]
        vals = marg[lo - 1:hi - 1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals))
            bc.append((lo + hi) / 2)
    if len(bm) >= 4:
        s, intercept, r, p, _ = stats.linregress(np.log(bc), np.log(bm))
        return s, r
    return None, None


print('Functions ready')

In [ ]:
# === Delete empty cached files ===
for f in RESULTS.glob('*.json'):
    with open(f) as fh:
        data = json.load(fh)
    if len(data) == 0:
        os.remove(f)
        print(f'Deleted empty: {f.name}')
print('Cache cleaned')

In [ ]:
# === Main run ===

doc_cache = {}


def get_tokens(doc_id):
    if doc_id not in doc_cache:
        fp = tok_manifest.get(doc_id)
        if fp is None:
            return None
        dp = to_drive_path(fp)
        if not Path(dp).exists():
            print(f'  File not found: {dp}')
            return None
        text = Path(dp).read_text(encoding='utf-8', errors='replace').strip()
        doc_cache[doc_id] = tokenizer.encode(text, add_special_tokens=False)
    return doc_cache[doc_id]


for corpus_id in RUN_CORPORA:
    cache_path = RESULTS / f'{corpus_id}.json'
    if cache_path.exists():
        with open(cache_path) as f:
            cached = json.load(f)
        if len(cached) > 0:
            print(f'\n{corpus_id}: cached ({len(cached)})')
            continue

    targets = targets_by_corpus.get(corpus_id, [])
    if not targets:
        print(f'\n{corpus_id}: no targets')
        continue

    print(f'\n{"="*50}')
    print(f'{corpus_id} ({len(targets)} targets)')
    print(f'{"="*50}')

    by_doc = {}
    for t in targets:
        by_doc.setdefault(t['document_id'], []).append(t)

    t0 = time.time()
    results = []
    doc_cache.clear()
    errors = 0

    for doc_id in tqdm(by_doc, desc=corpus_id):
        full_ids = get_tokens(doc_id)
        if full_ids is None:
            errors += 1
            continue

        for t in by_doc[doc_id]:
            cs = t['context_start_token']
            ce = t['context_end_token']
            ts = t['target_start_token']
            te = t['target_end_token']

            if ce > len(full_ids) or te > len(full_ids):
                continue

            ctx = full_ids[cs:ce]
            tgt = full_ids[ts:te]

            if len(ctx) < C or len(tgt) < 5:
                continue

            r = compute_corrected_curves(ctx, tgt)
            r['corpus_id'] = corpus_id
            r['document_id'] = doc_id
            r['target_id'] = t['target_id']
            r['target_frac'] = t['target_position_fraction']
            r['language'] = t['language']
            r['genre'] = t['genre']
            r['modality'] = t['modality']
            results.append(r)

    with open(cache_path, 'w') as f:
        json.dump(results, f)

    elapsed = time.time() - t0
    print(f'  {len(results)} results in {elapsed/60:.1f} min ({errors} load errors)')

    if results:
        curve = np.mean([r['delta_ppl'] for r in results], axis=0)
        total = np.mean(curve)
        alpha, r_val = fit_pl(np.array(curve))
        a_str = f'{alpha:.3f} (r={r_val:.3f})' if alpha else 'fit failed'
        print(f'  Mean Delta: {total:.4f}, alpha: {a_str}')

print('\nAll done!')

In [ ]:
# === Summary ===

print(f'{"Corpus":<30} {"N":>5} {"TotalDelta":>10} {"alpha":>8} {"r":>8}')
print('-' * 65)

for corpus_id in sorted(RUN_CORPORA):
    cp = RESULTS / f'{corpus_id}.json'
    if not cp.exists():
        continue
    with open(cp) as f:
        results = json.load(f)
    if not results:
        print(f'{corpus_id:<30} {0:>5} — empty')
        continue
    curve = np.mean([r['delta_ppl'] for r in results], axis=0)
    total = np.mean(curve)
    alpha, r_val = fit_pl(np.array(curve))
    a_str = f'{alpha:.3f}' if alpha else '—'
    r_str = f'{r_val:.3f}' if r_val else '—'
    print(f'{corpus_id:<30} {len(results):>5} {total:>10.4f} {a_str:>8} {r_str:>8}')

print(f'\nOriginal Wiki range (Llama): alpha ~ -0.73 to -0.88')

In [ ]:
# === Plots ===
import matplotlib.pyplot as plt

completed = []
for corpus_id in sorted(RUN_CORPORA):
    cp = RESULTS / f'{corpus_id}.json'
    if not cp.exists():
        continue
    with open(cp) as f:
        results = json.load(f)
    if results:
        completed.append((corpus_id, results))

if completed:
    n = len(completed)
    ncols = min(4, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)
    axes_flat = axes.flatten()

    for idx, (corpus_id, results) in enumerate(completed):
        ax = axes_flat[idx]
        curve = np.mean([r['delta_ppl'] for r in results], axis=0)
        smooth = uniform_filter1d(curve, 5)
        ax.plot(range(1, len(smooth) + 1), smooth, linewidth=2)
        ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
        alpha, r_val = fit_pl(np.array(curve))
        title = corpus_id.replace('_', ' ')
        if alpha:
            title += f'\nalpha={alpha:.2f} (r={r_val:.2f})'
        ax.set_title(title, fontweight='bold', fontsize=10)
        ax.set_xlabel('Distance d')
        ax.set_ylabel('Corrected Delta')
        ax.grid(True, alpha=0.15)

    for idx in range(n, len(axes_flat)):
        axes_flat[idx].set_visible(False)

    plt.suptitle(f'Corpus Expansion Phase {PHASE}',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS / f'fig_phase{PHASE}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved')
else:
    print('No completed corpora to plot')